In [ ]:
import os
import numpy as np
import soundfile as sf
from scipy.signal import correlate
from scipy.io import loadmat

from enfEstimationN import enfEstimationN
from filterN import filterN

In [ ]:
ref_folder = 'H1_ref_one_day'
h1_folder  = 'H1'

h1RefFileList = sorted([f for f in os.listdir(ref_folder) if f.endswith('.wav')])
h1FileList    = sorted([f for f in os.listdir(h1_folder) if f.endswith('.wav')])

refIndex  = loadmat("refIndex.mat")["refIndex"]
refPoints = loadmat("refPoints.mat")["refPoints"]

h1RefOneDayENF = []
h1ENF          = []

In [ ]:
wSize     = 32
hSize     = 1
timeShift = 30

fe        = 50
tol       = 0.5
nFFT      = 2**16

In [ ]:
for i, currentFileName in enumerate(h1RefFileList):
    os.system('cls' if os.name == 'nt' else 'clear')

    fullPath = os.path.join(ref_folder, currentFileName)

    print(f'\n ENF estimating: {currentFileName}\n')

    x, fsR = sf.read(fullPath)

    if len(x.shape) > 1:
        x = x[:, 0]

    enf = enfEstimationN(x, wSize, hSize, fe, nFFT, tol, fsR)
    h1RefOneDayENF.append(enf)
    
    del x

for i, currentFileName in enumerate(h1FileList):
    os.system('cls' if os.name == 'nt' else 'clear')

    fullPath = os.path.join(h1_folder, currentFileName)

    print(f'\n{currentFileName} ENF estimating.\n')

    x, fsH = sf.read(fullPath)

    if len(x.shape) > 1:
        x = x[:, 0]

    X = filterN(x, fsH, fsR, 2 * fe, tol)
    del x

    enf = enfEstimationN(X, wSize, hSize, 2 * fe, nFFT, tol, fsR) / 2
    h1ENF.append(enf)

    del X

In [ ]:
refPointsN = np.zeros((len(h1ENF), 2))
unMatchN   = []

m = 1
for i in range(len(h1ENF)):

    print(f'\n Calculating: {i+1}\n')

    signal1 = h1ENF[i].flatten()
    signal2 = h1RefOneDayENF[int(refIndex[i, 0]) - 1].flatten()

    corrM = correlate(signal2, signal1, mode='full')

    corrM = corrM / (np.linalg.norm(signal1) * np.linalg.norm(signal2))

    start_idx = len(signal1) - 1
    end_idx   = len(corrM) - len(signal1) + 1

    valid_corr = corrM[start_idx:end_idx]

    max_index = np.argmax(valid_corr)
    max_value = np.max(valid_corr)

    refPointsN[i, 0] = max_value
    refPointsN[i, 1] = max_index + 1

    if abs((refPoints[i, 1] / hSize) - refPointsN[i, 1]) >= (timeShift / hSize):
        unMatchN.append(i + 1)
        m += 1

    os.system('cls' if os.name == 'nt' else 'clear')

accuracy = ((len(h1ENF) - m) * 100 / len(h1ENF))
print(f'\n Accuracy: {accuracy:.2f} / 100 \n')